# max-back-tied-half — worked example 1: Demonstrate Mass Conservation of the Half-Mass Tie Convention

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `max-back-tied-half`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When differentiating `maximum(x, y)`, the gradient mass assigned to each input must sum to `grad_out` at every position. The half-mass convention assigns 1 to the strictly winning input, 0 to the loser, and 0.5 to both at ties. This guarantees `back0 + back1 = grad_out` everywhere, which is the conservation invariant — it mirrors the forward operation picking exactly one value at each position.

## Worked solution

**Step 1 — construct x and y with deliberate tie positions.**
We create two tensors where some positions have `x > y`, some have `x < y`, and some have `x == y` exactly. Using exact equality requires setting specific values rather than random draws.

**Step 2 — compute back0 and back1 using the half-mass rule.**
For back0 (gradient w.r.t. x): the mask is `(x > y) + 0.5*(x == y)`. For back1 (w.r.t. y): the mask is `(x < y) + 0.5*(x == y)`. Multiply each mask by `grad_out`.

**Step 3 — verify mass conservation.**
We check that `back0 + back1 == grad_out` at every position using `torch.allclose`. This confirms the half-mass split correctly distributes exactly the total incoming gradient.

**Step 4 — count the three cases.**
We print how many positions fell into each case (x wins, y wins, tie) and verify that the tie positions have exactly `0.5 * grad_out` in both back0 and back1.

In [ ]:
import torch as t

t.manual_seed(51)

def maximum_back0(grad_out, x, y):
    mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

def maximum_back1(grad_out, x, y):
    mask = (x < y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

# Construct tensors with known tie positions
x = t.tensor([3.0, 1.0, 2.0, 5.0, 2.0])
y = t.tensor([1.0, 4.0, 2.0, 3.0, 2.0])  # positions 2 and 4 are ties
grad_out = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0])

g0 = maximum_back0(grad_out, x, y)
g1 = maximum_back1(grad_out, x, y)

print("x:        ", x.tolist())
print("y:        ", y.tolist())
print("grad_out: ", grad_out.tolist())
print("back0:    ", g0.tolist())
print("back1:    ", g1.tolist())
print("back0+back1:", (g0 + g1).tolist())
print(f"\nMass conserved: {t.allclose(g0 + g1, grad_out, atol=1e-6)}")

n_ties = int((x == y).sum())
n_x_wins = int((x > y).sum())
n_y_wins = int((x < y).sum())
print(f"x wins: {n_x_wins}, y wins: {n_y_wins}, ties: {n_ties}")

# At tie positions, both get exactly half
tie_mask = (x == y)
print(f"Tie positions — back0: {g0[tie_mask].tolist()}")
print(f"Tie positions — back1: {g1[tie_mask].tolist()}")
print(f"Each is 0.5 * grad_out at ties: {t.allclose(g0[tie_mask], 0.5 * grad_out[tie_mask])}")